In [1]:
import os
import pandas as pd

In [2]:
with open('/u/rfechner/data/eic_gsm8k_deduplicated/test.parquet', 'rb') as file:
    queries = pd.read_parquet(file)

with open('/ptmp/rfechner/out/generated_behaviours/ReasoningStrategy_gsm8k__Qwen--Qwen3-8B.jsonl', 'r') as file:
    rt = pd.read_json(file, lines=True)

In [3]:
categories = {
            "Pattern Recognition": {
                "description": "Identifying regularities, repetitions, or structures within numbers, figures, or operations. Example: A student lists the first few triangular numbers (1, 3, 6, 10, \u2026) and notices that each is obtained by adding consecutive integers, leading to the conjecture T_n = n(n+1)/2."
            },
            "Backtracking": {
                "description": "Trying partial solutions and undoing steps when they lead to contradictions or dead ends; systematic trial-and-error with revision. Example: While solving a Sudoku puzzle, a student places numbers tentatively, backtracks when a rule is violated, and explores alternate paths. In algebra, testing possible integer roots of a polynomial and rejecting invalid ones."
            },
            "Simplification": {
                "description": "Reducing a complex problem to a simpler or specific case to reveal an underlying structure. Example: To prove a formula for the sum of the first n squares, the student first tries small n (e.g., n=1,2,3) to observe and test a conjectured pattern."
            },
            "Decomposition": {
                "description": "Breaking a problem into smaller, manageable subproblems or intermediate goals. Example: To solve a multi-step geometry proof, the student identifies subgoals like proving triangles ABC and DEF are similar before tackling the final ratio result."
            },
            "Verification": {
                "description": "Checking the correctness and coherence of a solution or conjecture. Example: After solving a system of equations, the student substitutes results into the original equations to confirm validity."
            },
            "Trials": {
                "description": "Testing multiple approaches or examples to gather insight or eliminate possibilities. Example: When unsure how to find integer solutions to x^2 + y^2 = 25, a student tries values of x (e.g., 0, 3, 4) until consistent pairs (3,4), (4,3), (0,5), etc. emerge."
            },
            "Analogical Mapping": {
                "description": "Using similarities between a known problem and a new one to transfer a solution method. Example: A student recalls solving 'sum of first n odd numbers' to find a pattern for 'sum of first n even numbers.'"
            }
        }

In [4]:
queries.reset_index(drop=True, inplace=True)
rt.reset_index(drop=True, inplace=True)
queries = queries.head(len(rt))

In [5]:
queries.iloc[0]['prompt'], rt.iloc[0]['data'][0]

('John decides to start collecting art. He pays the same price for his first 3 pieces of art and the total price came to $45,000. The next piece of art was 50% more expensive than those. How much did all the art cost?',
 'I notice a pattern in the pricing: the first three pieces have equal cost ($45,000 total means $15,000 each). This uniform pricing pattern breaks for the fourth piece, which is 50% more expensive ($22,500). Recognizing this shift from uniform pricing to a percentage-based increase reveals the total cost structure. The final answer is \\boxed{67500}.')

In [6]:
queries['augmented'] = rt['data']
queries['valid'] = queries['augmented'].apply(lambda x: len(x) == len(categories))
queries = queries[queries['valid']]

In [11]:
queries.head(2)

,prompt,correct_answer,incorrect_answer,correct_solution,incorrect_solution,explanation,error_type,wrong_step,source_file,line_idx,augmented,valid
0,John decides to start collecting art. He pays ...,67500.0,69750.0,"The first 3 pieces each cost 45000/3=$15,000\n...","The first 3 pieces each cost 45000/3=$15,000\n...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,0,[I notice a pattern in the pricing: the first ...,True
1,Irene earns $500 if she works for 40 hours a w...,700.0,750.0,"If Irene worked 50 hours last week, the total ...","If Irene worked 50 hours last week, the total ...",Step 3 adds information that is not mentioned ...,Hallucination,3,data/generated_cases_GSM8K/adding_irrelevant_i...,1,[I notice a pattern: Irene's income has two co...,True


In [8]:
entries = []
behaviours = list(categories.keys())
for i, row in queries.iterrows():
    augmented, original = \
        [{
            'prompt' : [{'role' : 'user', 'content' : row['prompt']}, 
                        {'role' : 'assistant', 'content' : row['augmented'][k]}],
            'behaviour_type' : behaviours[k],
            'index' : i
        } for k in range(len(behaviours))], \
        {
            'prompt' : [{'role' : 'user', 'content' : row['prompt']}, 
                        {'role' : 'assistant', 'content' : row['correct_solution']}],
            'behaviour_type' : 'anchor',
            'index' : i
        }
    
    entries.extend([*augmented, original])

In [9]:
out_df = pd.DataFrame(entries)

In [10]:
path = "/u/rfechner/data/incomplete_reasoning_strategies"
os.makedirs(path, exist_ok=True)

with open(os.path.join(path, 'deltas.jsonl'), 'w') as file:
    out_df.to_json(path_or_buf=file, lines=True, orient='records')